# 02 - Data Cleaning and Feature Preparation

This notebook prepares the collected Old School RuneScape Hiscores dataset for later similarity analysis and clustering.

The main goal is to transform the raw Hiscores output into a clean object-variable data matrix, where:

- each row represents one OSRS player,
- each column represents a numeric attribute,
- the selected attributes describe the player's skill profile.

This step is necessary before applying distance-based and clustering methods.

## 1. Package imports

In this section, we import the Python packages required for data loading, cleaning and basic feature preparation.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## 2. Project paths

The cleaned dataset will be read from and saved into the `data/processed` folder.

The previous notebook created the first processed CSV file from the collected Hiscores data.

In [2]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

input_path = PROCESSED_DATA_DIR / "osrs_hiscores_sample.csv"
output_path = PROCESSED_DATA_DIR / "osrs_hiscores_cleaned.csv"

input_path

WindowsPath('C:/Projects/osrs-player-segmentation/data/processed/osrs_hiscores_sample.csv')

## 3. Load the collected Hiscores dataset

The dataset contains public Hiscores data for selected OSRS players.

At this stage, the dataset contains several types of columns:

- player name,
- skill rank,
- skill level,
- skill experience.

For clustering, we will initially focus on skill level values, because they describe player progression in a simple and interpretable way.

In [3]:
df = pd.read_csv(input_path)

df.head()

,player,overall_rank,overall_level,overall_xp,attack_rank,attack_level,attack_xp,defence_rank,defence_level,defence_xp,...,runecraft_xp,hunter_rank,hunter_level,hunter_xp,construction_rank,construction_level,construction_xp,sailing_rank,sailing_level,sailing_xp
0,Zezima,1571605,1466,27957906,1556666,76,1343681,1420448,76,1342072,...,57263,1917011,45,62118,1895720,45,66561,-1,1,0
1,eszcape,317297,2107,201501042,251680,99,14852643,293228,99,13541188,...,1455752,478810,82,2505555,600507,83,2710608,330098,60,297498
2,Lynx Titan,103734,2278,4600000000,15,99,200000000,28,99,200000000,...,200000000,3,99,200000000,4,99,200000000,-1,1,0
3,Hey Jase,103735,2278,4600000000,19,99,200000000,58,99,200000000,...,200000000,34,99,200000000,10,99,200000000,-1,1,0
4,Woox,66972,2340,507712649,58399,99,26061798,33458,99,25478462,...,14386618,42409,99,15556190,32651,99,13400250,304792,63,394149


In [4]:
df.shape

(10, 76)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 76 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   player              10 non-null     object
 1   overall_rank        10 non-null     int64 
 2   overall_level       10 non-null     int64 
 3   overall_xp          10 non-null     int64 
 4   attack_rank         10 non-null     int64 
 5   attack_level        10 non-null     int64 
 6   attack_xp           10 non-null     int64 
 7   defence_rank        10 non-null     int64 
 8   defence_level       10 non-null     int64 
 9   defence_xp          10 non-null     int64 
 10  strength_rank       10 non-null     int64 
 11  strength_level      10 non-null     int64 
 12  strength_xp         10 non-null     int64 
 13  hitpoints_rank      10 non-null     int64 
 14  hitpoints_level     10 non-null     int64 
 15  hitpoints_xp        10 non-null     int64 
 16  ranged_rank         10 non-nu

## 4. Select skill level attributes

The Hiscores dataset contains rank, level and experience values for each skill.

In this notebook, we select only the columns ending with `_level`.

Reason:

- levels are easier to interpret than raw XP,
- all skill levels are on a similar scale,
- the result is suitable for a first clustering experiment.

In [6]:
level_columns = [column for column in df.columns if column.endswith("_level")]

level_columns

['overall_level',
 'attack_level',
 'defence_level',
 'strength_level',
 'hitpoints_level',
 'ranged_level',
 'prayer_level',
 'magic_level',
 'cooking_level',
 'woodcutting_level',
 'fletching_level',
 'fishing_level',
 'firemaking_level',
 'crafting_level',
 'smithing_level',
 'mining_level',
 'herblore_level',
 'agility_level',
 'thieving_level',
 'slayer_level',
 'farming_level',
 'runecraft_level',
 'hunter_level',
 'construction_level',
 'sailing_level']

In [7]:
df_levels = df[["player"] + level_columns].copy()

df_levels.head()

,player,overall_level,attack_level,defence_level,strength_level,hitpoints_level,ranged_level,prayer_level,magic_level,cooking_level,...,mining_level,herblore_level,agility_level,thieving_level,slayer_level,farming_level,runecraft_level,hunter_level,construction_level,sailing_level
0,Zezima,1466,76,76,75,1,1,1,1,80,...,63,52,70,53,71,45,44,45,45,1
1,eszcape,2107,99,99,99,99,99,82,99,99,...,81,87,78,80,99,99,76,82,83,60
2,Lynx Titan,2278,99,99,99,99,99,99,99,99,...,99,99,99,99,99,99,99,99,99,1
3,Hey Jase,2278,99,99,99,99,99,99,99,99,...,99,99,99,99,99,99,99,99,99,1
4,Woox,2340,99,99,99,99,99,99,99,99,...,99,99,99,99,99,99,99,99,99,63


## 5. Rename columns

The original column names contain the `_level` suffix.

For easier analysis, we remove this suffix and keep only the skill names.

In [8]:
df_levels.columns = [
    column.replace("_level", "") if column != "player" else column
    for column in df_levels.columns
]

df_levels.head()

,player,overall,attack,defence,strength,hitpoints,ranged,prayer,magic,cooking,...,mining,herblore,agility,thieving,slayer,farming,runecraft,hunter,construction,sailing
0,Zezima,1466,76,76,75,1,1,1,1,80,...,63,52,70,53,71,45,44,45,45,1
1,eszcape,2107,99,99,99,99,99,82,99,99,...,81,87,78,80,99,99,76,82,83,60
2,Lynx Titan,2278,99,99,99,99,99,99,99,99,...,99,99,99,99,99,99,99,99,99,1
3,Hey Jase,2278,99,99,99,99,99,99,99,99,...,99,99,99,99,99,99,99,99,99,1
4,Woox,2340,99,99,99,99,99,99,99,99,...,99,99,99,99,99,99,99,99,99,63


## 6. Basic data quality checks

Before feature engineering, we inspect the dataset for:

- missing values,
- invalid values,
- duplicate player names.

This is important because clustering algorithms are sensitive to noisy or inconsistent input data.

In [9]:
df_levels.isna().sum()

player          0
overall         0
attack          0
defence         0
strength        0
hitpoints       0
ranged          0
prayer          0
magic           0
cooking         0
woodcutting     0
fletching       0
fishing         0
firemaking      0
crafting        0
smithing        0
mining          0
herblore        0
agility         0
thieving        0
slayer          0
farming         0
runecraft       0
hunter          0
construction    0
sailing         0
dtype: int64

In [10]:
(df_levels == -1).sum()

player          0
overall         0
attack          0
defence         0
strength        0
hitpoints       0
ranged          0
prayer          0
magic           0
cooking         0
woodcutting     0
fletching       0
fishing         0
firemaking      0
crafting        0
smithing        0
mining          0
herblore        0
agility         0
thieving        0
slayer          0
farming         0
runecraft       0
hunter          0
construction    0
sailing         0
dtype: int64

In Hiscores data, `-1` can indicate that a value is not ranked or not available.

For skill level values this should be rare, but it is still safer to check and handle it.

In [11]:
df_levels = df_levels.replace(-1, np.nan)

df_levels.isna().sum()

player          0
overall         0
attack          0
defence         0
strength        0
hitpoints       0
ranged          0
prayer          0
magic           0
cooking         0
woodcutting     0
fletching       0
fishing         0
firemaking      0
crafting        0
smithing        0
mining          0
herblore        0
agility         0
thieving        0
slayer          0
farming         0
runecraft       0
hunter          0
construction    0
sailing         0
dtype: int64

## 7. Handle missing values

For this first version of the project, rows with missing values are removed.

This keeps the dataset simple and avoids introducing artificial player profiles through imputation.

In [12]:
df_levels = df_levels.dropna()

df_levels.shape

(10, 26)

## 8. Remove duplicate players

Each row should represent exactly one unique OSRS player.

Duplicate rows could distort similarity calculations and clustering results.

In [13]:
df_levels["player"].duplicated().sum()

0

In [14]:
df_levels = df_levels.drop_duplicates(subset="player")

df_levels.shape

(10, 26)

## 9. Create skill group features

The original skill columns describe individual OSRS skills.

To make the player profiles easier to interpret, we create aggregated skill group features.

These engineered features are not required for clustering, but they help with later interpretation of the clusters.

In [15]:
combat_skills = [
    "attack",
    "strength",
    "defence",
    "hitpoints",
    "ranged",
    "prayer",
    "magic"
]

gathering_skills = [
    "mining",
    "fishing",
    "woodcutting",
    "hunter",
    "sailing"
]

production_skills = [
    "cooking",
    "fletching",
    "firemaking",
    "crafting",
    "smithing",
    "herblore",
    "runecraft",
    "construction"
]

support_skills = [
    "agility",
    "thieving",
    "slayer",
    "farming"
]

In [16]:
df_levels["combat_score"] = df_levels[combat_skills].mean(axis=1)
df_levels["gathering_score"] = df_levels[gathering_skills].mean(axis=1)
df_levels["production_score"] = df_levels[production_skills].mean(axis=1)
df_levels["support_score"] = df_levels[support_skills].mean(axis=1)

df_levels.head()

,player,overall,attack,defence,strength,hitpoints,ranged,prayer,magic,cooking,...,slayer,farming,runecraft,hunter,construction,sailing,combat_score,gathering_score,production_score,support_score
0,Zezima,1466,76,76,75,1,1,1,1,80,...,71,45,44,45,45,1,33.000000,52.8,56.250,59.75
1,eszcape,2107,99,99,99,99,99,82,99,99,...,99,99,76,82,83,60,96.571429,76.8,86.375,89.00
2,Lynx Titan,2278,99,99,99,99,99,99,99,99,...,99,99,99,99,99,1,99.000000,79.4,99.000,99.00
3,Hey Jase,2278,99,99,99,99,99,99,99,99,...,99,99,99,99,99,1,99.000000,79.4,99.000,99.00
4,Woox,2340,99,99,99,99,99,99,99,99,...,99,99,99,99,99,63,99.000000,91.8,99.000,99.00


## 10. Create general progression features

In addition to skill group scores, we create general progression indicators:

- average skill level,
- highest skill level,
- lowest skill level,
- skill level standard deviation.

The standard deviation can indicate whether a player has a balanced or specialized account.

In [17]:
base_skill_columns = [
    column for column in df_levels.columns
    if column not in [
        "player",
        "overall",
        "combat_score",
        "gathering_score",
        "production_score",
        "support_score"
    ]
]

In [18]:
df_levels["average_skill_level"] = df_levels[base_skill_columns].mean(axis=1)
df_levels["max_skill_level"] = df_levels[base_skill_columns].max(axis=1)
df_levels["min_skill_level"] = df_levels[base_skill_columns].min(axis=1)
df_levels["skill_level_std"] = df_levels[base_skill_columns].std(axis=1)

df_levels.head()

,player,overall,attack,defence,strength,hitpoints,ranged,prayer,magic,cooking,...,construction,sailing,combat_score,gathering_score,production_score,support_score,average_skill_level,max_skill_level,min_skill_level,skill_level_std
0,Zezima,1466,76,76,75,1,1,1,1,80,...,45,1,33.000000,52.8,56.250,59.75,49.333333,99,1,31.568651
1,eszcape,2107,99,99,99,99,99,82,99,99,...,83,60,96.571429,76.8,86.375,89.00,87.791667,99,60,10.640281
2,Lynx Titan,2278,99,99,99,99,99,99,99,99,...,99,1,99.000000,79.4,99.000,99.00,94.916667,99,1,20.004166
3,Hey Jase,2278,99,99,99,99,99,99,99,99,...,99,1,99.000000,79.4,99.000,99.00,94.916667,99,1,20.004166
4,Woox,2340,99,99,99,99,99,99,99,99,...,99,63,99.000000,91.8,99.000,99.00,97.500000,99,63,7.348469


## 11. Inspect the cleaned dataset

Now we inspect the cleaned and enriched dataset with descriptive statistics.

This helps us understand the scale and distribution of the selected attributes before applying similarity and clustering methods.

In [19]:
df_levels.describe()

,overall,attack,defence,strength,hitpoints,ranged,prayer,magic,cooking,woodcutting,...,construction,sailing,combat_score,gathering_score,production_score,support_score,average_skill_level,max_skill_level,min_skill_level,skill_level_std
count,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,...,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.0,10.00000,10.000000
mean,2218.300000,96.700000,96.700000,96.600000,89.200000,89.200000,87.500000,89.200000,97.100000,95.100000,...,92.000000,46.900000,92.157143,84.420000,93.400000,93.925000,91.254167,99.0,46.30000,12.223224
std,275.880026,7.273239,7.273239,7.589466,30.990321,30.990321,30.858998,30.990321,6.008328,8.252272,...,17.262677,43.467612,20.799709,14.231561,13.637138,12.406344,15.092672,0.0,42.70324,10.511431
min,1466.000000,76.000000,76.000000,75.000000,1.000000,1.000000,1.000000,1.000000,80.000000,78.000000,...,45.000000,1.000000,33.000000,52.800000,56.250000,59.750000,49.333333,99.0,1.00000,0.000000
25%,2278.000000,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,...,99.000000,1.000000,99.000000,79.400000,98.531250,97.875000,94.916667,99.0,1.00000,3.066844
50%,2300.000000,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,...,99.000000,52.500000,99.000000,83.800000,99.000000,99.000000,95.833333,99.0,52.50000,10.831492
75%,2356.500000,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,...,99.000000,90.000000,99.000000,96.750000,99.000000,99.000000,98.187500,99.0,85.50000,20.004166
max,2376.000000,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,...,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,99.0,99.00000,31.568651


In [20]:
df_levels.head()

,player,overall,attack,defence,strength,hitpoints,ranged,prayer,magic,cooking,...,construction,sailing,combat_score,gathering_score,production_score,support_score,average_skill_level,max_skill_level,min_skill_level,skill_level_std
0,Zezima,1466,76,76,75,1,1,1,1,80,...,45,1,33.000000,52.8,56.250,59.75,49.333333,99,1,31.568651
1,eszcape,2107,99,99,99,99,99,82,99,99,...,83,60,96.571429,76.8,86.375,89.00,87.791667,99,60,10.640281
2,Lynx Titan,2278,99,99,99,99,99,99,99,99,...,99,1,99.000000,79.4,99.000,99.00,94.916667,99,1,20.004166
3,Hey Jase,2278,99,99,99,99,99,99,99,99,...,99,1,99.000000,79.4,99.000,99.00,94.916667,99,1,20.004166
4,Woox,2340,99,99,99,99,99,99,99,99,...,99,63,99.000000,91.8,99.000,99.00,97.500000,99,63,7.348469


## 12. Prepare feature matrix preview

For later similarity calculation and clustering, we need a numeric feature matrix.

The `player` column is an identifier, therefore it will not be used as a model input.

In [21]:
feature_columns = [
    column for column in df_levels.columns
    if column != "player"
]

X = df_levels[feature_columns]

X.head()

,overall,attack,defence,strength,hitpoints,ranged,prayer,magic,cooking,woodcutting,...,construction,sailing,combat_score,gathering_score,production_score,support_score,average_skill_level,max_skill_level,min_skill_level,skill_level_std
0,1466,76,76,75,1,1,1,1,80,78,...,45,1,33.000000,52.8,56.250,59.75,49.333333,99,1,31.568651
1,2107,99,99,99,99,99,82,99,99,81,...,83,60,96.571429,76.8,86.375,89.00,87.791667,99,60,10.640281
2,2278,99,99,99,99,99,99,99,99,99,...,99,1,99.000000,79.4,99.000,99.00,94.916667,99,1,20.004166
3,2278,99,99,99,99,99,99,99,99,99,...,99,1,99.000000,79.4,99.000,99.00,94.916667,99,1,20.004166
4,2340,99,99,99,99,99,99,99,99,99,...,99,63,99.000000,91.8,99.000,99.00,97.500000,99,63,7.348469


In [22]:
X.shape

(10, 33)

## 13. Save cleaned dataset

The cleaned dataset is saved into the `data/processed` folder.

This file will be used by the next notebooks:

- similarity and scaling analysis,
- clustering analysis.

In [23]:
df_levels.to_csv(output_path, index=False, encoding="utf-8")

output_path

WindowsPath('C:/Projects/osrs-player-segmentation/data/processed/osrs_hiscores_cleaned.csv')

## Summary

In this notebook, the collected OSRS Hiscores dataset was cleaned and prepared for further analysis.

The main steps were:

- loading the collected Hiscores dataset,
- selecting skill level attributes,
- checking invalid and missing values,
- removing duplicate players,
- creating aggregated skill group features,
- creating general progression indicators,
- saving the cleaned dataset.

The next notebook will focus on similarity, dissimilarity, scaling and distance-based analysis.